# Wczytanie plików Parquet z rzutowaniem dat

Notebook do Lakehouse w Microsoft Fabric. Wczytuje każdy plik Parquet z folderu w `Files`, zamienia kolumny typu TIMESTAMP_NTZ na `date` i zapisuje wynik jako tabelę Delta.

Modele semantyczne Direct Lake nie obsługują TIMESTAMP_NTZ, zob. https://learn.microsoft.com/fabric/fundamentals/delta-lake-interoperability

Przed uruchomieniem dołącz do notebooka docelowy Lakehouse jako domyślny i ustaw parametry w następnej komórce. Notebook nie był uruchamiany w Fabric, sprawdzono tylko logikę rzutowania w lokalnym PySpark.

In [ ]:
# Parametry
FOLDER = "Files/dane"      # folder z plikami .parquet w domyślnym Lakehouse
PREFIKS_TABELI = ""        # opcjonalny prefiks nazw tabel, np. "stg_"
TRYB_ZAPISU = "overwrite"  # overwrite albo append

In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampNTZType


def rzutuj_daty(df):
    """Zamienia kolumny TIMESTAMP_NTZ na date, inne kolumny zostawia bez zmian."""
    for pole in df.schema.fields:
        if isinstance(pole.dataType, TimestampNTZType):
            df = df.withColumn(pole.name, F.to_date(F.col(pole.name)))
    return df


def nazwa_tabeli(nazwa_pliku):
    """Load to Tables i Delta w Lakehouse przyjmują małe litery, cyfry i podkreślenie."""
    baza = re.sub(r"\.parquet$", "", nazwa_pliku, flags=re.IGNORECASE)
    return PREFIKS_TABELI + re.sub(r"[^a-z0-9_]", "_", baza.lower())

In [ ]:
pliki = [p for p in notebookutils.fs.ls(FOLDER) if p.name.lower().endswith(".parquet")]
print(f"Plików Parquet: {len(pliki)}")

for p in pliki:
    tabela = nazwa_tabeli(p.name)
    df = rzutuj_daty(spark.read.parquet(p.path))
    df.write.format("delta").mode(TRYB_ZAPISU).option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"{p.name} -> {tabela}")
    spark.table(tabela).printSchema()